<a href="https://colab.research.google.com/github/siya-pathak/from-scratch/blob/main/jev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q transformers datasets accelerate scikit-learn pandas

In [2]:
from datasets import load_dataset

train = load_dataset("Heliosoph/Jigsaw-Toxic-Comments", data_files="train.csv.gz", split="train")
print(train[0])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/8.74k [00:00<?, ?B/s]

train.csv.gz: reconstructing file:   0%|          |  0.00B / 26.7MB            

train.csv.gz: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

{'id': '0000997932d777bf', 'comment_text': "Explanation\nWhy the edits made under my username Hardcore Metallica Fan were reverted? They weren't vandalisms, just closure on some GAs after I voted at New York Dolls FAC. And please don't remove the template from the talk page since I'm retired now.89.205.38.27", 'toxic': 0, 'severe_toxic': 0, 'obscene': 0, 'threat': 0, 'insult': 0, 'identity_hate': 0}


In [3]:
test_text = load_dataset("Heliosoph/Jigsaw-Toxic-Comments", data_files={"test":"test.csv.gz"})

test.csv.gz: reconstructing file:   0%|          |  0.00B / 23.7MB            

test.csv.gz: downloading bytes:           |  0.00B            

Generating test split: 0 examples [00:00, ? examples/s]

In [4]:
test_labels = load_dataset("Heliosoph/Jigsaw-Toxic-Comments", data_files={"test":"test_labels.csv.gz"})

test_labels.csv.gz: reconstructing file:   0%|          |  0.00B / 1.29MB            

test_labels.csv.gz: downloading bytes:           |  0.00B            

Generating test split: 0 examples [00:00, ? examples/s]

Join the test text and test tabels and hugging face dataset had a very complicated way so using pandas instead
later converting df to hugging face dataset

In [5]:
import pandas as pd
from datasets import Dataset

text_df = test_text['test'].to_pandas()
labels_df = test_labels['test'].to_pandas()

test_df = text_df.merge(
    labels_df,
    on="id",
    how="inner",
    validate="one_to_one",
    indicator=True,
)


test_dataset = Dataset.from_pandas(test_df, preserve_index=False)
print(test_dataset)
print(test_dataset[0])
print(test_dataset.features)

Dataset({
    features: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate', '_merge'],
    num_rows: 153164
})
{'id': '00001cee341fdb12', 'comment_text': "Yo bitch Ja Rule is more succesful then you'll ever be whats up with you and hating you sad mofuckas...i should bitch slap ur pethedic white faces and get you to kiss my ass you guys sicken me. Ja rule is about pride in da music man. dont diss that shit on him. and nothin is wrong bein like tupac he was a brother too...fuckin white boys get things right next time.,", 'toxic': -1, 'severe_toxic': -1, 'obscene': -1, 'threat': -1, 'insult': -1, 'identity_hate': -1, '_merge': 'both'}
{'id': Value('string'), 'comment_text': Value('string'), 'toxic': Value('int64'), 'severe_toxic': Value('int64'), 'obscene': Value('int64'), 'threat': Value('int64'), 'insult': Value('int64'), 'identity_hate': Value('int64'), '_merge': Value('string')}


Per dataset documentation, labels are jointly missing in test data specifically, all can be -1 for some datapoints. Drop those

In [6]:
label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

test_dataset = test_dataset.filter(
    lambda row: all(row[col] != -1 for col in label_cols)
)

print(len(test_dataset))
print({col: sum(test_dataset[col]) for col in label_cols})

Filter:   0%|          | 0/153164 [00:00<?, ? examples/s]

63978
{'toxic': 6090, 'severe_toxic': 367, 'obscene': 3691, 'threat': 211, 'insult': 3427, 'identity_hate': 712}


In [7]:
from collections import Counter
for label in label_cols:
  print(label)
  print(Counter(train[label]))

toxic
Counter({0: 144277, 1: 15294})
severe_toxic
Counter({0: 157976, 1: 1595})
obscene
Counter({0: 151122, 1: 8449})
threat
Counter({0: 159093, 1: 478})
insult
Counter({0: 151694, 1: 7877})
identity_hate
Counter({0: 158166, 1: 1405})


X_train, Y_train, X_val, Y_val

In [8]:
split = train.train_test_split(test_size=0.2, shuffle=True, seed=42)
print(split)

DatasetDict({
    train: Dataset({
        features: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'],
        num_rows: 127656
    })
    test: Dataset({
        features: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'],
        num_rows: 31915
    })
})


In [9]:
x_train = split["train"]["comment_text"]
y_train = split["train"].select_columns(label_cols)
x_val = split["test"]["comment_text"]
y_val = split["test"].select_columns(label_cols)

In [10]:
print(len(x_train), len(y_train))
print(len(x_val), len(y_val))
print(type(y_train.column_names))

127656 127656
31915 31915
<class 'list'>


Loading the backbone of our JEV which is essentially just going to be our open source model.

From what I understand this is more or less like fine-tuning so our base model needs to stay frozen. Which means **the model needs to be in the eval mode** and we need to verify that no model params are trainable.

In [11]:
from pathlib import Path
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
CACHE_DIR = None
LABELS = tuple(y_train.column_names)
ROOT = Path.cwd()
HEAD_PATH = ROOT / "artifacts" / "jev_head.pt" #this will store the classification head

In [12]:
from transformers import AutoConfig, AutoModel
import torch

model = AutoModel.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    device_map="auto"
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [13]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, device_map="auto")

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [14]:
x_train[0]

"Missing Champions \n\nThis article should have ALL the TNA World Heavyweight Champions, from Ken Shamrock. Weird that it doesn't."

In [15]:
inputs = tokenizer(x_train[0], return_tensors="pt")
print(inputs)
input_ids = inputs['input_ids']
attention_mask = inputs['attention_mask']

{'input_ids': tensor([[25080, 28808,  4710,  1986,  4549,  1265,   614, 13097,   279,   350,
          7326,  4337, 28101,  4765, 28808,    11,   504, 14259, 72091, 20521,
            13, 76153,   429,   432,  3171,   944,    13]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1]])}


input_ids: Each number is a token ID representing a token from your comment. Shape (1, 27) means 1 comment with 27 tokens.



attention_mask: A 1 means the token is valid; a 0 means padding that the model should ignore. Here, all 27 tokens are valid.

In [16]:
import torch
from torch import nn

In [17]:
input_device = model.get_input_embeddings().weight.device

input_ids = input_ids.to(input_device)
attention_mask = attention_mask.to(input_device)

model.eval()
for parameter in model.parameters():
    parameter.requires_grad = False

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
        return_dict=True
    )

print(outputs)

BaseModelOutputWithPast(last_hidden_state=tensor([[[-3.4688, -1.3672,  0.3242,  ...,  1.6953,  2.6250, -0.7188],
         [-3.0312,  1.9219,  0.5625,  ...,  2.2656,  1.3594, -0.1934],
         [-0.8750,  1.4375,  1.9609,  ...,  1.9219, -1.2891,  0.1748],
         ...,
         [-1.1719, -0.9336,  4.1875,  ...,  0.1445, -2.2500, -0.7812],
         [-0.4980,  1.2734,  2.0000,  ...,  1.2266,  0.2910, -1.9219],
         [-1.0938, -0.1079,  3.3750,  ...,  0.0510,  1.7031, -0.3301]]],
       device='cuda:0', dtype=torch.bfloat16), past_key_values=None, hidden_states=None, attentions=None)


In [18]:
print(outputs.last_hidden_state.shape)

torch.Size([1, 27, 1536])


1: one comment.



27: 27 tokens in that comment.



1536: each token is represented by a 1536-dimensional contextual vector.


Now we need to pool this
**Because your task is to classify an entire comment, not individual tokens.**
The model gives you 27 vectors, one per token. But your classifier needs a fixed-size representation of the comment to predict labels like toxic, threat, and insult.
Pooling combines those 27 vectors into one 1536-dimensional vector representing the comment. You can then use that representation to predict the six labels.
Think of it as converting token-level information → comment-level information.

**Why do we use the attention mask?** Some comments are shorter than others. When processing comments in batches, shorter ones may be padded with extra tokens.

In [19]:
positions = attention_mask.sum(dim=1) - 1 #we take the last real token, so basically along dimension one (the 2nd dimension), we sum all the ones
#this is the sum of all the real tokens in the sentence (some sentences can have paddings making them have attention mask of 0)
#in our above case all are real tokens so we have positions = 27 - 1(indexing in python starts form 0)
positions = positions.clamp(min=0).long() #in case we have no attention mask, min position has to be 0, converted to long

We are using last vector pooling, assuming the last vector will have the full knowledge of the sentence

In [20]:
hidden = outputs.last_hidden_state #last update after the inputs have gone through the transformer
rows = torch.arange(hidden.size(0), device=hidden.device) #indices are on the same device (CPU or GPU) as the hidden states. #hidden.size(0) is the batch size

In [21]:
print(hidden.shape)

torch.Size([1, 27, 1536])


In [22]:
print(rows)

tensor([0], device='cuda:0')


In [23]:
print(positions)

tensor([26], device='cuda:0')


In [24]:
pooled = hidden[rows, positions] #so for each of the rows, each of the comments, we take the last token of dimension 1536

In [25]:
print(hidden[0, -1, :].shape) #we are using these 1536 and looking at the states one last time before we classify them into one of the multiple label

torch.Size([1536])



(B, 1536) -> (B, 6)


- \(B\): number of comments in a batch.
- 1536: features in each comment’s pooled representation.
- 6: one output score for each toxicity label.

In [26]:
linear = torch.nn.Linear(1536, 6)

In [27]:
linear = linear.to(pooled.device)

logits = linear(pooled.float())
print(logits.shape)  # (B, 6)

torch.Size([1, 6])


In [28]:
#logits and labels shape must match
y = list(y_train[0].values()) #we convert the label to a list to match with logits

BCEWithLogitsLoss calculates a binary classification loss for each of the 6 outputs, then averages the losses by default.

In [29]:
loss_fn = torch.nn.BCEWithLogitsLoss()
labels = torch.tensor(y, dtype=torch.float32).unsqueeze(0) #add the extra 1, dimension to match logits
labels = labels.to(logits.device)
loss = loss_fn(logits, labels)
print(loss.item())

1.4639252424240112


In [30]:
optimizer = torch.optim.AdamW(linear.parameters(), lr=1e-3)

optimizer.zero_grad()
loss.backward()
optimizer.step()

In [31]:
from torch.utils.data import DataLoader, TensorDataset

BATCH_SIZE = 128
EPOCHS = 3
LR = 1e-3
MAX_LENGTH = 256

In [33]:
print(type(x_train))
print(type(x_train[0]))
print(x_train[0])

<class 'datasets.arrow_dataset.Column'>
<class 'str'>
Missing Champions 

This article should have ALL the TNA World Heavyweight Champions, from Ken Shamrock. Weird that it doesn't.


In [34]:
train_tokens = tokenizer(
    list(x_train),
    truncation=True,
    max_length=MAX_LENGTH #We chose 256 as a starting point, not a universal optimum. Some comments may lose information if they're truncated, so we can revisit this choice later.
)

In [35]:
val_tokens = tokenizer(
    list(x_val), truncation=True, max_length=MAX_LENGTH
)

In [36]:
print(type(val_tokens))

<class 'transformers.tokenization_utils_base.BatchEncoding'>


In [37]:
train_ds = Dataset.from_dict(train_tokens)
val_ds = Dataset.from_dict(val_tokens)

In [38]:
train_ds = train_ds.add_column(
    "labels",
    [[y_train[i][c] for c in label_cols]
     for i in range(len(y_train))]
)
val_ds = val_ds.add_column(
    "labels",
    [[y_val[i][c] for c in label_cols]
     for i in range(len(y_val))]
)

In [39]:
from transformers import DataCollatorWithPadding
#helper that makes all comments in a batch the same length by adding padding tokens.
padder = DataCollatorWithPadding(tokenizer=tokenizer)

In [40]:
print(train_ds.column_names)

['input_ids', 'attention_mask', 'labels']


collate all the features(input ids and attention masks) and labels into a single dictionary called batch

In [41]:
def collate_fn(examples): #we send a batch of comments from train_ds
  features = {"input_ids": [], "attention_mask": []}
  label = []
  for example in examples:
    features["input_ids"].append(example["input_ids"])
    features["attention_mask"].append(example["attention_mask"])
    label.append(example['labels'])
  batch = padder(features)
  batch["labels"] = torch.tensor(label, dtype=torch.float32)
  return batch

In [42]:
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE,
    shuffle=True, collate_fn=collate_fn,
    num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE,
    shuffle=False, collate_fn=collate_fn,
    num_workers=2, pin_memory=True
)

In [43]:
#freeeze the base model
model.eval()
for p in model.parameters():
    p.requires_grad = False

In [44]:
print(model.config)

Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max_window_layers": 21,
  "model_type": "qwen2",
  

In [45]:
head_device = model.get_input_embeddings().weight.device #just to know where our embedding layer weights are
#classifier will go on the same place as the base model

In [46]:
linear_layer = torch.nn.Linear(model.config.hidden_size, len(label_cols))
linear_layer = linear_layer.to(head_device)

In [47]:
optimizer = torch.optim.AdamW(linear_layer.parameters(), lr=LR)
loss_fn = torch.nn.BCEWithLogitsLoss()

In [48]:
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm

# ========================================================
# STEP 1: Fast Feature Extraction (Pre-computing LLM Representations)
# ========================================================
@torch.no_grad()
def extract_features(loader):
    model.eval()
    all_features = []
    all_labels = []

    for batch in tqdm(loader, desc="Extracting features"):
        input_device = model.get_input_embeddings().weight.device
        input_ids = batch["input_ids"].to(input_device)
        attention_mask = batch["attention_mask"].to(input_device)
        labels = batch["labels"].to(head_device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True
        )
        hidden = outputs.last_hidden_state

        mask = attention_mask.to(hidden.device)
        positions = mask.sum(dim=1).sub(1).clamp(min=0).long()
        rows = torch.arange(hidden.size(0), device=hidden.device)
        pooled = hidden[rows, positions]

        all_features.append(pooled.cpu().float())
        all_labels.append(labels.cpu())

    return torch.cat(all_features, dim=0), torch.cat(all_labels, dim=0)

print("Extracting training features...")
X_train_pooled, Y_train_labels = extract_features(train_loader)

print("Extracting validation features...")
X_val_pooled, Y_val_labels = extract_features(val_loader)

# ========================================================
# STEP 2: Fast Training on Extracted Tensors
# ========================================================
# Create tiny, fast datasets from the pre-computed CPU/GPU-ready tensors
fast_train_ds = TensorDataset(X_train_pooled, Y_train_labels)
fast_val_ds = TensorDataset(X_val_pooled, Y_val_labels)

fast_train_loader = DataLoader(fast_train_ds, batch_size=512, shuffle=True)
fast_val_loader = DataLoader(fast_val_ds, batch_size=512, shuffle=False)

# Reset optimizer for the linear layer
linear_layer = torch.nn.Linear(model.config.hidden_size, len(label_cols)).to(head_device)
optimizer = torch.optim.AdamW(linear_layer.parameters(), lr=1e-3)
loss_fn = torch.nn.BCEWithLogitsLoss()

for epoch in range(EPOCHS):
    linear_layer.train()
    total_loss = 0.0

    for features, labels in fast_train_loader:
        features, labels = features.to(head_device), labels.to(head_device)

        logits = linear_layer(features)
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # Validation
    linear_layer.eval()
    val_loss = 0.0
    with torch.no_grad():
        for features, labels in fast_val_loader:
            features, labels = features.to(head_device), labels.to(head_device)
            logits = linear_layer(features)
            val_loss += loss_fn(logits, labels).item()

    print(
        f"Epoch {epoch + 1}/{EPOCHS} Finished | "
        f"Train loss: {total_loss / len(fast_train_loader):.4f} | "
        f"Val loss: {val_loss / len(fast_val_loader):.4f}"
    )

Extracting training features...


Extracting features:   0%|          | 0/998 [00:00<?, ?it/s]

Extracting validation features...


Extracting features:   0%|          | 0/250 [00:00<?, ?it/s]

Epoch 1/3 Finished | Train loss: 0.0786 | Val loss: 0.0623
Epoch 2/3 Finished | Train loss: 0.0590 | Val loss: 0.0602
Epoch 3/3 Finished | Train loss: 0.0562 | Val loss: 0.0597


### Regularized Head with Early Stopping
We define a lightweight MLP head containing a `Dropout` layer for regularization and write a training loop that keeps track of the best validation loss to prevent overfitting.

In [51]:
import copy

class RegularizedClassificationHead(torch.nn.Module):
    def __init__(self, input_dim, output_dim, dropout_prob=0.3):
        super().__init__()
        self.dropout = torch.nn.Dropout(dropout_prob)
        self.out_proj = torch.nn.Linear(input_dim, output_dim)

    def forward(self, x):
        # Apply dropout to the pooled features
        x = self.dropout(x)
        return self.out_proj(x)

# Instantiate the improved regularized model
reg_linear_layer = RegularizedClassificationHead(
    input_dim=model.config.hidden_size,
    output_dim=len(label_cols),
    dropout_prob=0.3
).to(head_device)

optimizer = torch.optim.AdamW(reg_linear_layer.parameters(), lr=1e-3, weight_decay=0.01)
loss_fn = torch.nn.BCEWithLogitsLoss()

# Early Stopping setup
patience = 5
best_val_loss = float('inf')
best_model_wts = copy.deepcopy(reg_linear_layer.state_dict())
epochs_no_improve = 0
max_epochs = 100

for epoch in range(max_epochs):
    reg_linear_layer.train()
    total_loss = 0.0

    for features, labels in fast_train_loader:
        features, labels = features.to(head_device), labels.to(head_device)

        logits = reg_linear_layer(features)
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # Validation
    reg_linear_layer.eval()
    val_loss = 0.0
    with torch.no_grad():
        for features, labels in fast_val_loader:
            features, labels = features.to(head_device), labels.to(head_device)
            logits = reg_linear_layer(features)
            val_loss += loss_fn(logits, labels).item()

    average_train_loss = total_loss / len(fast_train_loader)
    average_val_loss = val_loss / len(fast_val_loader)

    print(
        f"Epoch {epoch + 1}/{max_epochs} | "
        f"Train Loss: {average_train_loss:.4f} | "
        f"Val Loss: {average_val_loss:.4f}"
    )

    # Check early stopping criteria
    if average_val_loss < best_val_loss:
        best_val_loss = average_val_loss
        best_model_wts = copy.deepcopy(reg_linear_layer.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"\nEarly stopping triggered after {epoch + 1} epochs.")
            print(f"Best Validation Loss achieved: {best_val_loss:.4f}")
            break

# Load the best weights back
reg_linear_layer.load_state_dict(best_model_wts)
print("Restored best weights successfully!")

Epoch 1/100 | Train Loss: 0.1054 | Val Loss: 0.0653
Epoch 2/100 | Train Loss: 0.0681 | Val Loss: 0.0608
Epoch 3/100 | Train Loss: 0.0653 | Val Loss: 0.0607
Epoch 4/100 | Train Loss: 0.0642 | Val Loss: 0.0598
Epoch 5/100 | Train Loss: 0.0639 | Val Loss: 0.0596
Epoch 6/100 | Train Loss: 0.0638 | Val Loss: 0.0602
Epoch 7/100 | Train Loss: 0.0632 | Val Loss: 0.0604
Epoch 8/100 | Train Loss: 0.0638 | Val Loss: 0.0598
Epoch 9/100 | Train Loss: 0.0635 | Val Loss: 0.0589
Epoch 10/100 | Train Loss: 0.0635 | Val Loss: 0.0597
Epoch 11/100 | Train Loss: 0.0639 | Val Loss: 0.0596
Epoch 12/100 | Train Loss: 0.0638 | Val Loss: 0.0590
Epoch 13/100 | Train Loss: 0.0638 | Val Loss: 0.0597
Epoch 14/100 | Train Loss: 0.0641 | Val Loss: 0.0594

Early stopping triggered after 14 epochs.
Best Validation Loss achieved: 0.0589
Restored best weights successfully!


### Threshold Calibration
We find the optimal threshold for each class using our validation set to maximize the F1-score, and then apply those thresholds to evaluate our test set.

In [53]:
from sklearn.metrics import precision_recall_curve

# 1. Get predictions (probabilities) on the Validation set
reg_linear_layer.eval()
val_logits = []
fast_val_loader_eval = DataLoader(fast_val_ds, batch_size=512, shuffle=False)

with torch.no_grad():
    for features, _ in fast_val_loader_eval:
        features = features.to(head_device)
        logits = reg_linear_layer(features)
        val_logits.append(logits.cpu())

val_logits = torch.cat(val_logits, dim=0)
val_probs = torch.sigmoid(val_logits).numpy()
val_targets = Y_val_labels.numpy()

# 2. Find optimal threshold for each class on validation set
optimal_thresholds = {}
print("=== Calculating Optimal Thresholds on Validation Set ===")
for idx, col in enumerate(label_cols):
    precision, recall, thresholds = precision_recall_curve(val_targets[:, idx], val_probs[:, idx])

    # Prevent division by zero
    f1_scores = np.zeros_like(thresholds)
    for i in range(len(thresholds)):
        if (precision[i] + recall[i]) > 0:
            f1_scores[i] = (2 * precision[i] * recall[i]) / (precision[i] + recall[i])

    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]
    optimal_thresholds[col] = best_threshold
    print(f"{col.upper()}: Best Validation Threshold = {best_threshold:.4f} (Validation F1 = {f1_scores[best_idx]:.4f})")

# 3. Apply calibrated thresholds to the Test Set and evaluate
print("\n=== Calibrated Evaluation Metrics on Test Set ===")
for idx, col in enumerate(label_cols):
    auc = roc_auc_score(test_targets[:, idx], test_probs[:, idx])
    thresh = optimal_thresholds[col]

    # Apply the custom calibrated threshold
    preds = (test_probs[:, idx] > thresh).astype(int)

    print(f"\nCategory: {col.upper()} (Threshold: {thresh:.4f} | ROC-AUC: {auc:.4f})")
    print(classification_report(test_targets[:, idx], preds, target_names=["Non-toxic", col], digits=4))

=== Calculating Optimal Thresholds on Validation Set ===
TOXIC: Best Validation Threshold = 0.2538 (Validation F1 = 0.7318)
SEVERE_TOXIC: Best Validation Threshold = 0.1355 (Validation F1 = 0.4106)
OBSCENE: Best Validation Threshold = 0.2932 (Validation F1 = 0.7231)
THREAT: Best Validation Threshold = 0.3850 (Validation F1 = 0.3924)
INSULT: Best Validation Threshold = 0.2791 (Validation F1 = 0.6682)
IDENTITY_HATE: Best Validation Threshold = 0.1610 (Validation F1 = 0.4181)

=== Calibrated Evaluation Metrics on Test Set ===

Category: TOXIC (Threshold: 0.2538 | ROC-AUC: 0.9331)
              precision    recall  f1-score   support

   Non-toxic     0.9657    0.9449    0.9552     57888
       toxic     0.5656    0.6813    0.6181      6090

    accuracy                         0.9198     63978
   macro avg     0.7656    0.8131    0.7866     63978
weighted avg     0.9276    0.9198    0.9231     63978


Category: SEVERE_TOXIC (Threshold: 0.1355 | ROC-AUC: 0.9702)
              precision    

### Evaluation on the Held-out Test Set
We will extract the pooled features for our test dataset and compute classification metrics (ROC-AUC, F1, Precision, and Recall) across all 6 toxicity categories.

In [54]:
import numpy as np
from sklearn.metrics import roc_auc_score, classification_report

# 1. Tokenize test dataset
test_tokens = tokenizer(
    list(test_dataset["comment_text"]),
    truncation=True,
    max_length=MAX_LENGTH
)

test_ds = Dataset.from_dict(test_tokens)
test_ds = test_ds.add_column(
    "labels",
    [[test_dataset[i][c] for c in label_cols] for i in range(len(test_dataset))]
)

test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE,
    shuffle=False, collate_fn=collate_fn,
    num_workers=2, pin_memory=True
)

# 2. Extract features for test set
print("Extracting features for the test set...")
X_test_pooled, Y_test_labels = extract_features(test_loader)

# 3. Predict using our trained classification head
reg_linear_layer.eval()
test_logits = []

fast_test_ds = TensorDataset(X_test_pooled)
fast_test_loader = DataLoader(fast_test_ds, batch_size=512, shuffle=False)

with torch.no_grad():
    for (features,) in fast_test_loader:
        features = features.to(head_device)
        logits = reg_linear_layer(features)
        test_logits.append(logits.cpu())

test_logits = torch.cat(test_logits, dim=0)
test_probs = torch.sigmoid(test_logits).numpy()
test_targets = Y_test_labels.numpy()

# 4. Calculate and display performance metrics using calibrated thresholds
print("\n=== Calibrated Evaluation Metrics on Test Set ===")
for idx, col in enumerate(label_cols):
    auc = roc_auc_score(test_targets[:, idx], test_probs[:, idx])
    # Use the validation-calibrated thresholds instead of 0.5
    thresh = optimal_thresholds[col]
    preds = (test_probs[:, idx] > thresh).astype(int)

    print(f"\nCategory: {col.upper()} (Threshold: {thresh:.4f} | ROC-AUC: {auc:.4f})")
    print(classification_report(test_targets[:, idx], preds, target_names=["Non-toxic", col], digits=4))

Extracting features for the test set...


Extracting features:   0%|          | 0/500 [00:00<?, ?it/s]


=== Evaluation Metrics on Test Set ===

Category: TOXIC (ROC-AUC: 0.9331)
              precision    recall  f1-score   support

   Non-toxic     0.9524    0.9741    0.9631     57888
       toxic     0.6856    0.5374    0.6025      6090

    accuracy                         0.9325     63978
   macro avg     0.8190    0.7558    0.7828     63978
weighted avg     0.9270    0.9325    0.9288     63978


Category: SEVERE_TOXIC (ROC-AUC: 0.9702)
              precision    recall  f1-score   support

   Non-toxic     0.9958    0.9961    0.9960     63611
severe_toxic     0.2857    0.2670    0.2761       367

    accuracy                         0.9920     63978
   macro avg     0.6407    0.6316    0.6360     63978
weighted avg     0.9917    0.9920    0.9918     63978


Category: OBSCENE (ROC-AUC: 0.9488)
              precision    recall  f1-score   support

   Non-toxic     0.9698    0.9870    0.9783     60287
     obscene     0.7008    0.4974    0.5818      3691

    accuracy                